<a href="https://colab.research.google.com/github/adalbertii/Track-optimization/blob/main/VRP_with_capacity_and_time_window_constraints_09_06_2026_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 wh

In [2]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import random

def create_data_model():
    """Stores the data for the problem."""
    data = {}
    num_nodes = 20
    depot_index = 0

    # Generate random locations for nodes
    locations = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(num_nodes)]

    # Calculate distance matrix
    distance_matrix = []
    for from_node in locations:
        row = []
        for to_node in locations:
            dist = int(((from_node[0] - to_node[0]) ** 2 + (from_node[1] - to_node[1]) ** 2) ** 0.5)
            row.append(dist)
        distance_matrix.append(row)

    # Demands for each node (excluding depot)
    demands = [0] + [random.randint(1, 10) for _ in range(1, num_nodes)]

    # Vehicle data
    vehicle_number = 4
    vehicle_capacities = [30 for _ in range(vehicle_number)]

    # Time windows for each node (start, end)
    time_windows = []
    for _ in range(num_nodes):
        start = random.randint(0, 50)
        end = start + random.randint(10, 50)
        time_windows.append((start, end))
    # Depot has time window 0..100
    time_windows[depot_index] = (0, 100)

    data['distance_matrix'] = distance_matrix
    data['demands'] = demands
    data['vehicle_capacities'] = vehicle_capacities
    data['num_vehicles'] = vehicle_number
    data['depot'] = depot_index
    data['time_windows'] = time_windows

    return data

def solve_vrp_with_constraints(data):
    """Solves the VRP with capacity and time window constraints."""
    # Create the routing index manager
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    # Create Routing Model
    routing = pywrapcp.RoutingModel(manager)

    # Create and register a transit callback (distance)
    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['distance_matrix'][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Add Capacity constraint
    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return data['demands'][from_node]

    demand_callback_index = routing.RegisterTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,  # null capacity slack
        data['vehicle_capacities'],  # vehicle maximum capacities
        True,  # start cumul to zero
        'Capacity'
    )

    # Add Time Windows constraint
    def time_callback(from_index, to_index):
        return distance_callback(from_index, to_index)

    time_callback_index = routing.RegisterTransitCallback(time_callback)

    routing.AddDimension(
        time_callback_index,
        30,  # allow waiting time
        100,  # maximum time per vehicle
        False,  # Don't force start cumul to zero
        'Time'
    )
    time_dimension = routing.GetDimensionOrDie('Time')

    # Add time window constraints for each node
    for node_idx, (start, end) in enumerate(data['time_windows']):
        index = manager.NodeToIndex(node_idx)
        time_dimension.CumulVar(index).SetRange(start, end)

    # Add time window constraints for each vehicle start node (depot)
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        time_dimension.CumulVar(index).SetRange(
            data['time_windows'][data['depot']][0],
            data['time_windows'][data['depot']][1]
        )

    # Setting search parameters
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    search_parameters.time_limit.FromSeconds(30)

    # Solve the problem
    solution = routing.SolveWithParameters(search_parameters)
    return (routing, manager, solution)

def print_solution(data, routing, manager, solution):
    """Prints solution on console."""
    if solution:
        total_distance = 0
        total_load = 0
        time_dimension = routing.GetDimensionOrDie('Time')
        for vehicle_id in range(data['num_vehicles']):
            index = routing.Start(vehicle_id)
            plan_output = f'Route for vehicle {vehicle_id}:\n'
            route_distance = 0
            route_load = 0
            while not routing.IsEnd(index):
                node_index = manager.IndexToNode(index)
                load = data['demands'][node_index]
                route_load += load
                time_var = time_dimension.CumulVar(index)
                plan_output += f' {node_index} (load={route_load}) Time={solution.Min(time_var)}..{solution.Max(time_var)} -> '
                previous_index = index
                index = solution.Value(routing.NextVar(index))
                route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
            node_index = manager.IndexToNode(index)
            plan_output += f' {node_index}\n'
            print(plan_output)
            print(f'Route distance: {route_distance}')
            print(f'Route load: {route_load}\n')
            total_distance += route_distance
        print(f'Total distance of all routes: {total_distance}')
    else:
        print('No solution found!')

def main():
    data = create_data_model()
    routing, manager, solution = solve_vrp_with_constraints(data)
    print_solution(data, routing, manager, solution)

if __name__ == '__main__':
    main()

Route for vehicle 0:
 0 (load=0) Time=0..100 ->  0

Route distance: 0
Route load: 0

Route for vehicle 1:
 0 (load=0) Time=0..100 ->  0

Route distance: 0
Route load: 0

Route for vehicle 2:
 0 (load=0) Time=0..25 ->  17 (load=2) Time=5..25 ->  13 (load=7) Time=11..25 ->  10 (load=17) Time=11..25 ->  9 (load=23) Time=11..25 ->  7 (load=26) Time=15..25 ->  6 (load=27) Time=15..25 ->  3 (load=36) Time=15..33 ->  1 (load=38) Time=15..40 ->  0

Route distance: 0
Route load: 38

Route for vehicle 3:
 0 (load=0) Time=6..40 ->  19 (load=6) Time=36..40 ->  18 (load=7) Time=37..40 ->  16 (load=16) Time=37..40 ->  15 (load=20) Time=40..40 ->  14 (load=21) Time=40..40 ->  12 (load=23) Time=40..40 ->  11 (load=28) Time=45..57 ->  8 (load=38) Time=45..68 ->  5 (load=44) Time=45..70 ->  4 (load=51) Time=45..80 ->  2 (load=55) Time=47..93 ->  0

Route distance: 0
Route load: 55

Total distance of all routes: 0
